# 🛡️ 06 — Robustness and T-GAE Anomaly Detection

Two questions: **can a temporal graph autoencoder (T-GAE) flag unusual market days**, and **how stable are the
contagion models when their inputs are noisy, incomplete or deliberately manipulated?**

1. T-GAE anomaly scores over time, checked against independently defined stress days (same day and next day).
2. Perturbations: input noise and missing events, TGN vs gradient boosting (test period).
3. Gradient attacks: white-box FGSM on the TGN, transferred to gradient boosting.
4. The most anomalous test days: matched to catalogued events, otherwise reported as "unclassified".

**Setup.** T-GAE (`models.tgae.SnapshotTGAE`): two graph-convolution layers over each day's correlation graph
(trailing 168 h, 23:00 UTC), a GRU across days, and decoders for the graph and each coin's activity that day. The
anomaly score is the **one-day-ahead prediction error**, i.e. how surprising a day is given the past. Prediction was
chosen on validation over plain reconstruction, after a first design using one hour's features showed no signal.
Trained on 2020–22 snapshots, 5 seeds. A **stress day** has at least one hour in which ≥ 70% of active coins shocked
(21% of days). This label comes from the returns, not from the T-GAE. Stress tests (`robustness/run.py`) use the final
TGN (seed 0) and the final boosting model on the 2024–26 test events; a clean pass first reproduces their saved
predictions.

In [1]:
import sys, json
from pathlib import Path
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch  # keep torch before pandas (Windows DLL order)
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_dark"
V2 = "../data/processed/v2"

snaps = torch.load(f"{V2}/1h/graph_snapshots.pt", weights_only=False)
print(f"{len(snaps['snapshot_t'])} hourly graph snapshots, threshold {snaps['undirected_threshold']}, "
      f"{len(snaps['graph_coins'])} coins — input for the T-GAE")

2456 hourly graph snapshots, threshold 0.75, 30 coins — input for the T-GAE


In [2]:
import glob
from sklearn.metrics import roc_auc_score

tg_runs = sorted(glob.glob("../runs/*_tgae_1h_final_*"))
mets = [json.loads(Path(r, "metrics.json").read_text()) for r in tg_runs]
sc = pd.concat([pd.read_csv(Path(r, "tgae_scores.csv")).assign(seed=i) for i, r in enumerate(tg_runs)])
avg = sc.groupby("time", as_index=False).agg(score=("score", "mean"), stress=("stress", "first"),
                                              stress_next_day=("stress_next_day", "first"), split=("split", "first"),
                                              turbulence=("turbulence", "first"), density=("density", "first"))
avg["time"] = pd.to_datetime(avg["time"])
print(f"{len(tg_runs)} T-GAE seeds; best epochs {[m['best_epoch'] for m in mets]}; stress days {avg['stress'].mean():.1%}")
runs = pd.read_csv("../runs/results.csv")
display(runs[runs["experiment"].isin(["tgae_1h_final", "robustness_1h"])][["run_id", "commit", "dirty", "status"]])

5 T-GAE seeds; best epochs [74, 48, 63, 93, 81]; stress days 20.7%


,run_id,commit,dirty,status
67,20261007-023753_robustness_1h_0_eb04c49,eb04c49,True,ok
68,20261007-023753_tgae_1h_final_0_eb04c49,eb04c49,True,ok
69,20261007-024440_tgae_1h_final_1_eb04c49,eb04c49,True,ok
70,20261007-024835_tgae_1h_final_2_eb04c49,eb04c49,True,ok
71,20261007-025815_tgae_1h_final_3_eb04c49,eb04c49,True,ok
72,20261007-034225_tgae_1h_final_4_eb04c49,eb04c49,True,ok


*Provenance:* these runs show `dirty = True` only because `docs/challenges_and_solutions.md` is kept out of version control on purpose; the code is exactly commit `eb04c49`. The robustness run's clean pass reproduced the TGN's saved test predictions (max |Δp| 2×10⁻⁴), and the refitted boosting model reproduced its saved validation AUPRC exactly.

## 1. T-GAE anomaly scores vs stress days

In [3]:
events = pd.read_csv("../data/external/events.csv", parse_dates=["date"])
fig = go.Figure()
fig.add_trace(go.Scatter(x=avg["time"], y=avg["score"].rolling(7, min_periods=1).mean(), name="T-GAE score (7-day mean)"))
st_days = avg[avg["stress"]]
fig.add_trace(go.Scatter(x=st_days["time"], y=[avg["score"].quantile(0.02)] * len(st_days), mode="markers",
                         marker=dict(size=3, color="#ff5252"), name="stress day"))
for _, e in events[events["date"] >= avg["time"].min()].iterrows():
    fig.add_vline(x=e["date"], line_dash="dot", line_color="#888")
for b in ("2022-07-01", "2024-01-01"):
    fig.add_vline(x=pd.Timestamp(b), line_color="#ffd54f")
fig.update_layout(title="T-GAE anomaly score (mean of 5 seeds; dotted = catalogued events, yellow = split boundaries)", height=430)
fig.show()
rows = []
for s in ("val", "test"):
    for target in ("stress", "stress_next_day"):
        r = {"split": s, "target": "same day" if target == "stress" else "next day"}
        for name in ("score", "turbulence", "density"):
            v = [m.get(f"{s}_auroc_{target}_{name}", np.nan) for m in mets]
            r[{"score": "T-GAE", "turbulence": "day turbulence", "density": "graph density"}[name]] = \
                f"{np.mean(v):.3f} ± {np.std(v, ddof=1):.3f}" if name == "score" else f"{v[0]:.3f}"
        rows.append(r)
display(pd.DataFrame(rows).set_index(["split", "target"]))
inc = pd.DataFrame([{"target": t, "turbulence only": np.mean([m[f"test_auroc_{t}_logit_turbulence"] for m in mets]),
                     "turbulence + T-GAE": np.mean([m[f"test_auroc_{t}_logit_turbulence_plus_score"] for m in mets])}
                    for t in ("stress", "stress_next_day")]).set_index("target")
print("Does the T-GAE add to turbulence? Logistic models fitted on validation, AUROC on test (mean of 5 seeds):")
display(inc.round(3))

T-GAE day turbulence graph density
split target                                              
val   same day  0.557 ± 0.004          0.857         0.594
      next day  0.519 ± 0.005          0.539         0.482
test  same day  0.511 ± 0.010          0.893         0.594
      next day  0.490 ± 0.008          0.600         0.512

Does the T-GAE add to turbulence? Logistic models fitted on validation, AUROC on test (mean of 5 seeds):


,turbulence only,turbulence + T-GAE
target,,
stress,0.893,0.898
stress_next_day,0.600,0.589


*What this shows:* **The T-GAE does not recognise market stress days.** Its AUROC for same-day stress is 0.511 on test (0.557 on
validation), close to chance. The day's plain turbulence (mean |z|) reaches 0.893. As a next-day warning the T-GAE
scores 0.490 against 0.600 for turbulence. Adding the T-GAE score to turbulence changes almost nothing (same day 0.893
→ 0.898, next day 0.600 → 0.589). The five seeds agree (±0.01), so this is not training noise. The reason: an
autoencoder learns to reproduce what it sees, extreme days included, so its error measures how *unusual the
structure* is, not how *turbulent* the day is. Two designs were tried on validation (single-hour features with
reconstruction, then daily activity with one-day-ahead prediction) before this final run; neither found a signal.
**For stress detection, simple turbulence measures are the right tool.**

## 2. Perturbations: noisy inputs and missing events

**Input noise:** Gaussian noise of standard deviation σ (inputs are standardised, so σ = 0.5 is half a typical
deviation) on the explained event's inputs: node features of both coins and the event message, for both models.
**Missing events:** a random share of test events never reaches the TGN's memory and neighbour lists (data-feed
gaps); every event is still scored. Gradient boosting has no event memory, so missing events cannot affect it.

In [4]:
stx = pd.read_csv("../reports/robustness/stress_tests.csv")
base = stx[stx["perturbation"] == "none"].set_index("model")
stx["rel_k1h"] = stx.apply(lambda r: r["auprc_k1h"] / base.loc[r["model"], "auprc_k1h"] - 1, axis=1)
stx["rel_k6h"] = stx.apply(lambda r: r["auprc_k6h"] / base.loc[r["model"], "auprc_k6h"] - 1, axis=1)
nm = stx[stx["perturbation"].isin(["none", "input noise", "missing events"])]
fig = px.line(nm[nm["perturbation"] != "none"], x="level", y="auprc_k1h", color="model", facet_col="perturbation",
              markers=True, title="Test AUPRC (k = 1 h) under perturbation", facet_col_spacing=0.08)
fig.update_xaxes(matches=None)
fig.update_layout(height=400)
fig.show()
display(nm[["model", "perturbation", "level", "auprc_k1h", "auprc_k6h", "auprc_k24h", "rel_k1h", "rel_k6h", "mean_abs_dp"]]
        .round(4).set_index(["perturbation", "level", "model"]).sort_index())

auprc_k1h  auprc_k6h  auprc_k24h  \
perturbation   level model                                                 
input noise    0.10  TGN                   0.2027     0.4632      0.7381   
                     gradient boosting     0.1772     0.4467      0.7310   
               0.25  TGN                   0.2000     0.4591      0.7352   
                     gradient boosting     0.1727     0.4392      0.7245   
               0.50  TGN                   0.1917     0.4469      0.7264   
                     gradient boosting     0.1694     0.4342      0.7147   
               1.00  TGN                   0.1710     0.4155      0.7049   
                     gradient boosting     0.1644     0.4257      0.7023   
missing events 0.10  TGN                   0.2036     0.4643      0.7388   
                     gradient boosting     0.1778     0.4511      0.7360   
               0.25  TGN                   0.2041     0.4650      0.7393   
                     gradient boosting     0.1778     0.4511      0.7360   
               0.50  TGN                   0.2037     0.4651      0.7398   
                     gradient boosting     0.1778     0.4511      0.7360   
none           0.00  TGN                   0.2032     0.4641      0.7387   
                     gradient boosting     0.1778     0.4511      0.7360   

                                        rel_k1h  rel_k6h  mean_abs_dp  
perturbation   level model                                             
input noise    0.10  TGN                -0.0025  -0.0018       0.0052  
                     gradient boosting  -0.0033  -0.0096       0.0314  
               0.25  TGN                -0.0160  -0.0106       0.0129  
                     gradient boosting  -0.0286  -0.0263       0.0488  
               0.50  TGN                -0.0568  -0.0371       0.0256  
                     gradient boosting  -0.0472  -0.0374       0.0650  
               1.00  TGN                -0.1587  -0.1046       0.0500  
                     gradient boosting  -0.0754  -0.0561       0.0825  
missing events 0.10  TGN                 0.0018   0.0005       0.0014  
                     gradient boosting   0.0000   0.0000       0.0000  
               0.25  TGN                 0.0041   0.0019       0.0021  
                     gradient boosting   0.0000   0.0000       0.0000  
               0.50  TGN                 0.0024   0.0021       0.0037  
                     gradient boosting   0.0000   0.0000       0.0000  
none           0.00  TGN                 0.0000   0.0000       0.0000  
                     gradient boosting   0.0000   0.0000       0.0000

*What this shows:* **Missing data does not hurt the TGN at all:** with 10%, 25% or even 50% of test events never reaching its memory
and neighbour lists, AUPRC stays within ±0.4% of the clean score. This confirms notebook 04: the TGN's predictions rest
on the coins' current state, not on specific past events. **Moderate noise affects both models about equally:** at σ =
0.5 the TGN loses 5.7% at 1 hour and boosting 4.7%. Only at very heavy noise (σ = 1, as large as the typical variation
of each input) does the TGN lose more (−16% vs −8% at 1 hour). Even then it scores higher than boosting at 1 hour
(0.171 vs 0.164), though not at 6 hours (0.416 vs 0.426). The TGN's predictions are steadier: under the same noise its
probabilities move by about half as much as boosting's (mean |Δp| 0.026 vs 0.065 at σ = 0.5).

## 3. Gradient attacks (FGSM)

White-box fast-gradient-sign attack on the TGN: every input of the explained event is moved by ±ε in the direction
that most increases the loss against the true labels. This is the worst case for a small, bounded manipulation of the
observed data by someone who knows the model. The same perturbed inputs are then fed to gradient boosting (a
**transfer** attack: the attacker does not know the boosting model).

In [5]:
fg = stx[stx["perturbation"].str.startswith("FGSM") | (stx["perturbation"] == "none")]
display(fg[["model", "perturbation", "level", "auprc_k1h", "auprc_k6h", "auprc_k24h", "rel_k1h", "rel_k6h", "mean_abs_dp"]]
        .round(4).set_index(["model", "level"]))
fig = px.line(fg[fg["perturbation"] != "none"], x="level", y="rel_k6h", color="model", markers=True,
              title="Relative change in test AUPRC (k = 6 h) under FGSM (white-box on TGN, transferred to boosting)")
fig.update_layout(height=380, yaxis_tickformat=".0%")
fig.show()

,,perturbation,auprc_k1h,auprc_k6h,auprc_k24h,rel_k1h,rel_k6h,mean_abs_dp
model,level,,,,,,,
TGN,0.00,none,0.2032,0.4641,0.7387,0.0000,0.0000,0.0000
gradient boosting,0.00,none,0.1778,0.4511,0.7360,0.0000,0.0000,0.0000
TGN,0.05,FGSM attack (white-box),0.1640,0.3808,0.6858,-0.1932,-0.1793,0.0109
gradient boosting,0.05,FGSM transferred from TGN,0.1889,0.4034,0.6776,0.0621,-0.1057,0.0285
TGN,0.10,FGSM attack (white-box),0.1313,0.3137,0.6378,-0.3541,-0.3240,0.0215
gradient boosting,0.10,FGSM transferred from TGN,0.1833,0.3727,0.6544,0.0306,-0.1737,0.0414
TGN,0.25,FGSM attack (white-box),0.0801,0.2127,0.5424,-0.6061,-0.5417,0.0500
gradient boosting,0.25,FGSM transferred from TGN,0.1534,0.3033,0.5953,-0.1374,-0.3276,0.0733


*What this shows:* **A deliberate attack is the TGN's weak point.** A white-box FGSM step of only 0.05 standard deviations per input
cuts its test AUPRC at 1 hour by 19%, at 0.1 by 35% and at 0.25 by 61% (0.203 → 0.080). Small, targeted changes to the
observed features of the two coins and the event can therefore mislead it, as is typical of neural networks.
Transferred to gradient boosting (an attacker who knows only the TGN), the same perturbations do not hurt it at 1
hour (it even scores slightly higher at small ε). They do hurt it at 6 hours (−11% to −33%), because they push the
same features (volatility, volume) in the same directions. **Implication:** a deployed system should not
rely on one differentiable model fed by data that an adversary can touch. Validating inputs against several
exchanges, combining with a non-differentiable model (the ensemble of notebook 03) and adversarial training are the
standard defences, left for future work. Random noise of the same size (σ = 0.1) costs almost nothing (−0.3%), so the
risk is manipulation, not ordinary data errors.

## 4. The most anomalous test days

The ten test days with the highest T-GAE score (mean of 5 seeds). A day is matched to a catalogued event when one
lies within ±2 days; otherwise it is reported as **unclassified**. No manipulation type is inferred: the T-GAE only
says a day was unusual, not why.

In [6]:
tst = avg[avg["split"] == "test"].nlargest(10, "score").copy()
def match(d):
    near = events[(events["date"] - d).abs() <= pd.Timedelta(days=2)]
    return near.iloc[0]["name"] if len(near) else "unclassified"
tst["catalogued event (±2 days)"] = tst["time"].map(match)
tst["score percentile"] = tst["score"].map(lambda v: (avg.loc[avg["split"] == "test", "score"] <= v).mean())
display(tst[["time", "score", "score percentile", "stress", "turbulence", "catalogued event (±2 days)"]].round(3))

,time,score,score percentile,stress,turbulence,catalogued event (±2 days)
2103,2025-10-10 23:00:00,21.329,1.000,True,1.990,Oct-10 liquidation cascade
2392,2026-07-26 23:00:00,14.429,0.999,False,0.742,unclassified
2393,2026-07-27 23:00:00,14.283,0.998,True,0.978,unclassified
2407,2026-08-10 23:00:00,14.026,0.997,False,0.790,unclassified
1508,2024-02-23 23:00:00,13.901,0.996,False,0.765,unclassified
2404,2026-08-07 23:00:00,13.767,0.995,False,0.631,unclassified
1519,2024-03-05 23:00:00,13.673,0.994,True,1.326,unclassified
2406,2026-08-09 23:00:00,13.446,0.993,False,0.706,unclassified
2403,2026-08-06 23:00:00,13.385,0.992,False,0.734,unclassified
2408,2026-08-11 23:00:00,13.232,0.991,False,0.781,unclassified


*What this shows:* **The single most anomalous test day is the 10-Oct-2025 liquidation cascade** (score 21.3 vs 14.4 for the next test day; over all periods only two validation days in June and August 2023 come close).
So the T-GAE does notice the most extreme structural break of the test period, even though it does not track ordinary
stress days. The next nine days cluster in two short periods (late February – early March 2024 and late July –
mid August 2026). Most of them were **not** stress days and had normal turbulence, and none is near a catalogued
event, so they are reported as **unclassified**. They mark days whose correlation structure or per-coin activity was
unusual given the past, which a human should look at; nothing in the data says whether they reflect manipulation,
listings, market-structure changes or chance, so no such label is given.

## Key Takeaways

- **The T-GAE is not a stress detector.** Its anomaly score barely separates market-wide stress days (test AUROC
  0.51) and adds nothing to simple turbulence (0.89) as a same-day or next-day signal. Two designs were tried on
  validation and both are reported. It does rank the 10-Oct-2025 cascade as the most anomalous test day.
- **The TGN is robust to missing data:** losing up to half of the test events changes its AUPRC by less than 0.5%.
- **Random noise hurts both models similarly** up to moderate levels, and the TGN's predictions are steadier
  (half the change of boosting's).
- **Deliberate gradient attacks are the main vulnerability:** a perturbation of 0.1 standard deviations cuts the TGN's
  AUPRC by about a third. Defences (input validation across sources, model ensembles, adversarial training) are
  future work.
- **Unusual days are reported as "unclassified":** no manipulation type is inferred from an anomaly score.